# Week1_ex3 - Mesh Convergence Verification (companion)

**Purpose**: check whether the adaptive mesh generated at the nominal `magnet_z0 = -1cm` point gets reused across the full 17-point position sweep. That would explain why `Force_z` bottoms out near zero right at -1cm and peaks symmetrically on either side of it (see `Week1_ex3_analysis.ipynb`, Hypothesis 1).

This notebook attaches to the already-solved Week1_ex3 project - no re-solve, no new geometry. It pulls the per-variation solve profile and checks the tetrahedra count.

**Note**: `list_of_variations()`'s exact keyword arguments (`setup_name`/`sweep_name`) haven't been run against this project yet either -- if it errors on the argument names, check the installed pyaedt 0.15.3 signature with `help(M3D.list_of_variations)` and adjust.

**Note**: the exact signature and output format of `GetProfile()` for a variation sweep hasn't been verified against this project yet. Run the last cell first and look at what `profile_raw` actually contains before trusting any parsing built on top of it.

**Note**: the project path is rebuilt the same way the original notebook saved it (`ANSYS_PROJECT_DIR` env var if set, otherwise the working directory this notebook is launched from). If it's opened from a different folder than where `Week1_ex3.ipynb` was run, set `ANSYS_PROJECT_DIR` first or this attach step will fail.

In [ ]:
# 1. Imports
import os
import pandas as pd
import ansys.aedt.core

# 2. Suppress screen logging so local paths don't leak into notebook output
ansys.aedt.core.settings.enable_screen_logs = False


In [ ]:
# 3. Attach to the already-solved Week1_ex3 project (no re-solve)
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 4. Rebuild the same path the original notebook saved to
# (ANSYS_PROJECT_DIR if set on this machine, otherwise the notebook's own working directory)
proj_name = "Week1_ex3"
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
project_path = os.path.join(base_dir, proj_name, f"{proj_name}.aedt")

# 5. Re-attach Maxwell3d to the existing project and design
M3D = ansys.aedt.core.Maxwell3d(project=project_path, design="Week1_ex3")


In [ ]:
# 6. Get the list of solved variation strings for the position sweep
# (using the wrapped PyAEDT method here, not raw odesign.GetVariations() -- that one failed over gRPC)
variations = M3D.list_of_variations(setup_name="Setup1", sweep_name="LastAdaptive")

# 7. Pull the raw solve profile for each variation
# unverified: confirm this actually returns per-variation text and includes a tetrahedra count
rows = []
for v in variations:
    profile_text = M3D.odesign.GetProfile("Setup1", v)
    rows.append({"variation": v, "profile_raw": profile_text})

df_profile = pd.DataFrame(rows)
df_profile
